In [38]:
import requests


In [194]:
BASE_URL = "https://api.mangadex.org"
title = "one piece"

# Search manga title

In [195]:
search_params = {
    "title": title,
    "limit": 10,
    "availableTranslatedLanguage[]": ["en", "ja", "ko"],
}
r_search = requests.get(f"{BASE_URL}/manga", params=search_params)
    
manga = r_search.json().get("data", [])
print(len(manga))

10


In [196]:
manga_id = manga[0]['id']
manga_id

'a1c7c817-4e59-43b7-9365-09675a149a6f'

# Get chapters

In [ ]:
feed_params = {
        "order[chapter]": "desc",
        "limit": 5,
        # "includeEmptyPages": 0,
        # "includeExternalUrl": 0
    }
r_feed = requests.get(f"{BASE_URL}/manga/{manga_id}/feed", params=feed_params)
chapters = r_feed.json().get("data", [])
chapters

[]

In [198]:
chapter_id = chapters[0]['id'] #first chapter (latest chapter)'s id
url = chapters[0]['attributes']['externalUrl']
chapters[0]

{'id': '414f2112-cfba-4e00-88f6-d5375122cbb8',
 'type': 'chapter',
 'attributes': {'volume': None,
  'chapter': '1175',
  'title': 'El drac dels llamps Nidhog',
  'translatedLanguage': 'ca',
  'externalUrl': None,
  'isUnavailable': False,
  'publishAt': '2026-03-08T18:55:23+00:00',
  'readableAt': '2026-03-08T18:55:23+00:00',
  'createdAt': '2026-03-08T18:55:22+00:00',
  'updatedAt': '2026-03-08T18:56:10+00:00',
  'version': 3,
  'pages': 12},
 'relationships': [{'id': 'ac77b807-c451-4f30-88d5-3c6e1d4ff2ff',
   'type': 'scanlation_group'},
  {'id': 'a1c7c817-4e59-43b7-9365-09675a149a6f', 'type': 'manga'},
  {'id': 'bf87bf74-000b-42d5-a146-cc7bc1229a7c', 'type': 'user'}]}

In [190]:
!mloader {url} -r

Usage: mloader [OPTIONS] [URLS]...
Try 'mloader --help' for help.

Error: Invalid value for '[URLS]...': Invalid url: None


# NOTE: If pages = 0 or theres an externalURL, it means the manga is not hosted on mangadex so we can't download the images!

In [199]:
def get_chapter_panels(id):
    # 1. Ask MangaDex which server to use
    r = requests.get(f"https://api.mangadex.org/at-home/server/{id}")
    data = r.json()
    if data['result'] == 'error':
        print("No chapter, probably hosted externally.")
        return []
    # 2. Grab the base URL and the chapter-specific hash
    base_url = data["baseUrl"]
    chapter_hash = data["chapter"]["hash"]
    
    # 'data' contains the high-quality filenames
    # 'dataSaver' contains the compressed/smaller filenames
    file_names = data["chapter"]["data"] 
    
    # 3. Construct the full URL for every page
    # Format: {baseUrl}/data/{hash}/{filename}
    urls = [f"{base_url}/data/{chapter_hash}/{name}" for name in file_names]
    
    return urls

In [200]:
panel_urls = get_chapter_panels(chapter_id)

if panel_urls:
    first_panel = panel_urls[0]
    first_panel

In [193]:
panel_urls[:3]

['https://cmdxd98sb0x3yprd.mangadex.network/data/f6ede24511b9ff7eeb165f84433ebf05/1-02a722c36bb229478caa9ed72a8ba850caa6902cae776f226ce99e6f0bab8e7f.png',
 'https://cmdxd98sb0x3yprd.mangadex.network/data/f6ede24511b9ff7eeb165f84433ebf05/2-1db2fd23857faea34c3944d8aad2d79c0103610f0a9ac1b6fd7b6bcba7c9a345.png',
 'https://cmdxd98sb0x3yprd.mangadex.network/data/f6ede24511b9ff7eeb165f84433ebf05/3-54ed76fc80a445e4f0da14a9331cd997b960dff604237aa058c052e645ca2bd3.png']

# downloading the chapter locally

In [12]:
def download_panel(url, filename="manga_panel.jpg"):
    # 1. Send a GET request to the image URL
    response = requests.get(url, stream=True)
    
    if response.status_code == 200:
        # 2. Open a local file in 'wb' (write binary) mode
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        print(f"Success! Saved as {filename}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")

In [14]:
test = "https://cmdxd98sb0x3yprd.mangadex.network/data-saver/4007d56744b3ae0b1dd54707fc4d780a/1-dc301bd32e67ad8e11113b4c2aa5626a2117c4463152408f62c0e7ad81a13bce.jpg"
download_panel(test, "test.jpg")

Success! Saved as test.jpg


# testing mloader to download frm mangaplus

In [36]:
url = chapters[0]['attributes']['externalUrl']
url

'https://mangaplus.shueisha.co.jp/viewer/1028362'

In [37]:
!mloader {url} -r


           _                 _
 _ __ ___ | | ___   __ _  __| | ___ _ __
| '_ ` _ \| |/ _ \ / _` |/ _` |/ _ \ '__|
| | | | | | | (_) | (_| | (_| |  __/ |
|_| |_| |_|_|\___/ \__,_|\__,_|\___|_|

13.03.2026 14:18:01 |   INFO   |  __main__.py   206  | Started export
13.03.2026 14:18:03 |   INFO   |   loader.py    140  | 1/1) Manga: One Piece
13.03.2026 14:18:03 |   INFO   |   loader.py    141  |     Author: Eiichiro Oda
13.03.2026 14:18:03 |   INFO   |   loader.py    152  |     1/1) Chapter #1176: Chapter 1176: With Pride
#1176
13.03.2026 14:18:26 |   INFO   |  __main__.py   227  | SUCCESS
